In [1]:
!pip install torch torchvision pillow gradio

In [2]:
# =========================================================
# FULL GRADIO DASHBOARD – Personalised UI Aesthetics Predictor
# =========================================================
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms, models
import gradio as gr

# ---------------------------
# 1. Paths and device
# ---------------------------
CSV_PATH = "/kaggle/input/datasets/rahulsonboro/ui-aestheticdataset/Aes_dataset - Sheet1.csv"
DESKTOP_IMG_DIR = "/kaggle/input/datasets/rahulsonboro/ui-aestheticdataset/images/desktop img"
MOBILE_IMG_DIR   = "/kaggle/input/datasets/rahulsonboro/ui-aestheticdataset/images/mobile img"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# 2. Load CSV and compute statistics (user means, global mean, client groups)
# ---------------------------
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.dropna(inplace=True)

# Per-user mean rating
user_means = {}
for uid, group in df.groupby('user_id'):
    all_ratings = pd.concat([group['desktop_rating'], group['mobile_rating']])
    user_means[uid] = all_ratings.mean()

GLOBAL_MEAN = pd.concat([df['desktop_rating'], df['mobile_rating']]).mean()

# Client grouping (same as during training)
def get_client_id(gender, qualification):
    gender_str = 'male' if gender == 0 else 'female'
    qual_map = {0: 'nonworking', 1: 'student', 2: 'working'}
    return f"{gender_str}_{qual_map[qualification]}"

user_client_map = {}
for _, row in df.iterrows():
    uid = row['user_id']
    if uid not in user_client_map:
        user_client_map[uid] = get_client_id(row['gender'], row['qualification'])

client_counts = {}
for uid, cid in user_client_map.items():
    client_counts[cid] = client_counts.get(cid, 0) + 1

# Average user_mean per client group
group_means = {}
for cid in client_counts:
    users_in_group = [uid for uid, c in user_client_map.items() if c == cid]
    group_means[cid] = np.mean([user_means[uid] for uid in users_in_group])

print("Group means:", group_means)

# ---------------------------
# 3. Model definition (must match training)
# ---------------------------
class EfficientNetRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.backbone(x).squeeze(1)

# ---------------------------
# 4. Load trained model
# ---------------------------
model = EfficientNetRegressor().to(DEVICE)
model.load_state_dict(torch.load("/kaggle/input/models/rahulsonboro/best-federated-model/other/default/1/best_federated_model.pth", map_location=DEVICE, weights_only=True))
model.eval()

# ---------------------------
# 5. Image preprocessing (same as validation transform)
# ---------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------------------
# 6. Prediction function
# ---------------------------
def predict_aesthetics(image, client_group):
    """
    image: PIL image uploaded via Gradio
    client_group: one of the demographic groups (or 'None' for global)
    """
    image = image.convert("RGB")
    img_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_normalized = model(img_tensor).item()   # in [0,1]

    # Use group‑mean if a valid group is selected, else fallback to global mean
    if client_group in group_means:
        group_mean = group_means[client_group]
    else:
        group_mean = GLOBAL_MEAN

    # Reverse the personalisation + min‑max scaling
    pred_original = pred_normalized * 9.0 + 1.0 + group_mean - GLOBAL_MEAN
    pred_original = max(1.0, min(10.0, pred_original))

    return round(pred_original, 2)

# ---------------------------
# 7. Gradio interface
# ---------------------------
client_options = list(group_means.keys())   # ['female_nonworking', 'female_student', ...]

iface = gr.Interface(
    fn=predict_aesthetics,
    inputs=[
        gr.Image(type="pil", label="Upload UI Screenshot"),
        gr.Dropdown(choices=client_options, value=client_options[0], label="Demographic Group")
    ],
    outputs=gr.Number(label="Predicted Aesthetic Score (1–10)"),
    title="UI Aesthetic Predictor (Federated EfficientNet-B0)",
    description="Select the demographic group to obtain a personalised aesthetic rating."
)

iface.launch(share=True)

Group means: {'female_student': np.float64(5.966285714285714), 'male_working': np.float64(5.8875), 'female_nonworking': np.float64(6.6000000000000005), 'male_student': np.float64(6.019200000000001), 'female_working': np.float64(6.65)}
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://40844fbc2a37e9064f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


========================================================================================================

==========================================================================================================

=======================================================================================================

# =========================================================
# GRADIO DASHBOARD – Federated EfficientNet-B0
# Loads best_federated_model.pth, predicts aesthetic score
# =========================================================
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import gradio as gr

# ---------------------------
# 1. Model definition (must match training)
# ---------------------------
class EfficientNetRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)   # we load our own weights
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.backbone(x).squeeze(1)

# ---------------------------
# 2. Load the trained model
# ---------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientNetRegressor().to(DEVICE)
model.load_state_dict(torch.load("/kaggle/input/models/rahulsonboro/best-federated-model/other/default/1/best_federated_model.pth", map_location=DEVICE, weights_only=True))
model.eval()

# ---------------------------
# 3. Preprocessing (same as validation transform)
# ---------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Global mean rating (from training) – used when user is unknown
GLOBAL_MEAN = 6.0613   # adjust if yours is different

# ---------------------------
# 4. Prediction function
# ---------------------------
def predict_aesthetics(image, user_id=""):
    """
    image: PIL image uploaded via Gradio
    user_id: optional – if you know the user, you could apply
             personalised centering. For now we use the global mean.
    """
    # Convert to RGB (in case it's RGBA / greyscale)
    image = image.convert("RGB")

    # Apply transforms
    img_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

    # Inference
    with torch.no_grad():
        pred_normalized = model(img_tensor).item()   # in [0,1]

    # Convert back to original 1–10 scale
    pred_original = pred_normalized * 9.0 + 1.0
    pred_original = max(1.0, min(10.0, pred_original))  # clip just in case

    return round(pred_original, 2)

# ---------------------------
# 5. Gradio Interface
# ---------------------------
iface = gr.Interface(
    fn=predict_aesthetics,
    inputs=gr.Image(type="pil", label="Upload UI Screenshot"),
    outputs=gr.Number(label="Predicted Aesthetic Score (1–10)"),
    title="UI Aesthetic Predictor",
    description="Upload a desktop or mobile UI image to get its aesthetic rating (1–10)."
)

# Launch the dashboard (share=True gives a public link)
iface.launch(share=True)